# <p style="color:#07d839; font-family:sans-serif;"> 😀 Sentiment Analysis — Análisis de Sentimiento 🥲 </p>

---

## <p style="color:#07d839;">🎯 Objetivo de la Tarea</p>

El propósito de este análisis es desentrañar la carga emocional oculta tras las letras de diferentes géneros musicales. Buscamos responder preguntas clave sobre la naturaleza sentimental de la música:

*   **Polaridad:** ¿Qué géneros son predominantemente positivos, negativos o neutros?
*   **Intensidad:** ¿Existen géneros con una carga emocional negativa o positiva mucho más marcada?
*   **Emociones específicas:** ¿Qué géneros destacan en sentimientos como la **tristeza**, **ira**, **alegría** o **amor**?
*   **Variabilidad:** ¿Es el sentimiento constante dentro de un género o existe mucha disparidad entre canciones?

---

## <p style="color:#07d839;">⚙️ Metodología del Análisis</p>

La unidad básica de estudio es **cada canción individual**. El flujo de procesamiento sigue esta estructura lógica:

> **Flujo de Trabajo:**
> `Canción` ➔ `Letra (Texto)` ➔ `Modelo de Sentimiento` ➔ `Score (Positivo/Negativo/Neutro)`

### Procesamiento por Género
Una vez obtenido el score individual, agruparemos los resultados para obtener una visión panorámica por categoría:

| Género Musical | Métricas a Calcular |
| :--- | :--- |
| **Pop** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **Rap** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **Rock** | Media de sentimiento, % Positivas, % Negativas, % Neutras |
| **...** | ... |



---

In [165]:
# librerías
import pandas as pd
import glob
import os
import numpy as np
import re
import ast
import html
from pathlib import Path


---
### 1. Limpieza dataset

+ Eliminar canciones sin lyrics.

+ Eliminar canciones sin ningún género válido.

+ Quitar letras demasiado cortas, por ejemplo menos de 30 o 50 palabras.

+ Normalizar nombres de géneros: pop, Pop, POP → pop.

+ Revisar duplicados por artist + song.

+ NLP sobre lyrics para preparar análisis de sentimiento 

Limpiar la letra sin destruir información útil para el sentimiento. En análisis de sentimiento no conviene eliminar stopwords agresivamente, porque palabras como not, never, no, without cambian completamente el significado.

In [ ]:
# Fusionar Datasets - !!!!! SOLO EJECUTAR UNA VEZ !!!!!!

patron_busqueda = "music_dataset_*.csv"
archivos_csv = glob.glob(patron_busqueda)

# Verificamos
if not archivos_csv:
    print("No se encontraron archivos con el prefijo 'music_dataset_'. Revisa el directorio.")
else:
    print(f"Archivos encontrados para fusionar: {archivos_csv}")
    
    # Leer cada archivo CSV y almacenarlo en una lista
    lista_dataframes = []
    
    for archivo in archivos_csv:
        # Leemos el CSV
        df = pd.read_csv(archivo)
        lista_dataframes.append(df)
        print(f"Leído: {archivo} con {len(df)} filas.")

    # Concatenar (fusionar) todos los DataFrames de la lista en uno solo
    # ignore_index=True es importante para que el índice se reinicie (0, 1, 2...) 
    # en lugar de mantener los índices originales de cada archivo por separado.
    df_final = pd.concat(lista_dataframes, ignore_index=True)

    # Guardar el resultado en un nuevo archivo CSV
    nombre_archivo_salida = "music_dataset_final.csv"
    
    # index=False evita que se guarde la columna de números de fila en el CSV final
    df_final.to_csv(nombre_archivo_salida, index=False)

    print(f"\n¡Fusión completada! El archivo final tiene {len(df_final)} filas en total.")
    print(f"Se ha guardado como: {nombre_archivo_salida}")

Archivos encontrados para fusionar: ['music_dataset_20000_30000.csv', 'music_dataset_40000_50000.csv', 'music_dataset_jose.csv']
Leído: music_dataset_20000_30000.csv con 1035 filas.
Leído: music_dataset_40000_50000.csv con 101 filas.
Leído: music_dataset_jose.csv con 201 filas.

¡Fusión completada! El archivo final tiene 1337 filas en total.
Se ha guardado como: music_dataset_final.csv


In [166]:
# cargar dataset
DATA_PATH = "music_dataset_final.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (1337, 7)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3
0,Tarja Turunen,Until Silence,NaN,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal
1,Susan Boyle,You Have To Be There,What is it Lord that you want\n\nThat I am not...,"pop, love it, female vocalists",pop,love it,female vocalists
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah)\n\n\n\nFever dream hi...","electropop, pop, synthpop",electropop,pop,synthpop
3,Taylor Swift,London Boy,[Idris Elba and James Corden:]\n\nWe can go dr...,NaN,NaN,NaN,NaN
4,Taylor Swift,Sugar,What a thing to see\n\nWhat a thing to be\n\nW...,NaN,NaN,NaN,NaN


,artist,song,lyrics,genres,genre_1,genre_2,genre_3
1332,Sarah Brightman,The Music of the Night,"Nighttime sharpens, heightens each sensation.\...","classical, female vocalists, opera",classical,female vocalists,opera
1333,The Phantom Of The Opera,Wandering Child,NaN,NaN,NaN,NaN,NaN
1334,Placido Domingo,Yesterday,NaN,"opera, classical, tenor",opera,classical,tenor
1335,Tarja Turunen,Goldfinger,NaN,"symphonic metal, female vocalists, finnish",symphonic metal,female vocalists,finnish
1336,Tarja Turunen,Until Silence,NaN,"symphonic metal, rock, tarja turunen",symphonic metal,rock,tarja turunen


In [167]:
# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

expected_cols = ["artist", "song", "lyrics", "genres", "genre_1", "genre_2", "genre_3"]

missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas en el dataset: {missing_cols}")

print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['artist', 'song', 'lyrics', 'genres', 'genre_1', 'genre_2', 'genre_3']


### 1.1 NLP

**PREPROCESADO DE LYRICS**

In [168]:
def clean_lyrics(text):
    """
    Limpieza conservadora de letras para análisis emocional.
    No elimina stopwords ni aplica stemming/lemmatization.
    """
    
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML
    text = html.unescape(text)
    
    # Eliminar URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas raras
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar etiquetas típicas de letras
    section_patterns = [
        r"\[.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\]",
        r"\(.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\)"
    ]
    
    for pattern in section_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    
    # Eliminar patrones frecuentes de Genius u otras fuentes
    text = re.sub(r"\d+\s*contributors?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"you might also like", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"embed$", " ", text, flags=re.IGNORECASE)
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")
    
    # Espacios múltiples
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

df["word_count"] = df["lyrics_clean"].apply(count_words)

display(df[["artist", "song", "lyrics_clean", "word_count"]].head())

,artist,song,lyrics_clean,word_count
0,Tarja Turunen,Until Silence,NaN,0
1,Susan Boyle,You Have To Be There,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
3,Taylor Swift,London Boy,[Idris Elba and James Corden:] We can go drivi...,443
4,Taylor Swift,Sugar,What a thing to see What a thing to be What a ...,241


- No hemos eliminado stopwords.
- No hemos hecho stemming.
- No hemos hecho lemmatization.
- No hemos eliminado signos de puntuación de forma agresiva.
- No hemos convertido todo a minúsculas.
- No hemos eliminado negaciones como "not", "no", "never".
- No hemos tokenizado manualmente para entrenar un modelo clásico.

En la fase de preprocesado de letras se aplicó una limpieza conservadora orientada al uso posterior de un modelo Transformer de clasificación emocional. Se eliminaron elementos no pertenecientes a la letra, como URLs, etiquetas estructurales de canciones ([Verse], [Chorus], [Bridge]), patrones procedentes del scraping y espacios duplicados. También se normalizaron comillas y entidades HTML para mejorar la consistencia del texto.

No se aplicaron técnicas agresivas como eliminación de stopwords, stemming o lematización, ya que el modelo emocional necesita conservar el contexto lingüístico completo de las frases. Finalmente, se calculó el número de palabras de cada letra y se descartaron canciones sin letra válida o con letras demasiado cortas.

**PREPROCESADO DE GENRES**

In [169]:
VALID_GENRES = {
    "alt country",
    "alt rock",
    "alt-country",
    "alternative",
    "alternative metal",
    "alternative rock",
    "ambient",
    "american country",
    "art rock",
    "black metal",
    "bluegrass",
    "blues",
    "blues rock",
    "blues-rock",
    "bossa nova",
    "canadian country",
    "celtic",
    "chicago blues",
    "chillout",
    "classic blues",
    "classic country",
    "classic rock",
    "classical",
    "contemporary country",
    "contry",
    "country",
    "country and western",
    "country pop",
    "country rap",
    "country rock",
    "country two step",
    "country-pop",
    "dance",
    "deep house",
    "delta blues",
    "disco",
    "dream pop",
    "electro house",
    "electro pop",
    "electroclash",
    "electronic",
    "electropop",
    "emo",
    "emocore",
    "experimental",
    "folk",
    "folk pop",
    "folk rock",
    "folkrock",
    "funk",
    "future bass",
    "gospel",
    "gothic metal",
    "hard rock",
    "hip hop",
    "honky tonk",
    "house",
    "indie",
    "indie folk",
    "indie pop",
    "indie rock",
    "indietronica",
    "jazz",
    "melodic death metal",
    "metal",
    "modern country",
    "neo-soul",
    "new age",
    "new traditionalist",
    "opera",
    "outlaw country",
    "pop",
    "pop country",
    "pop punk",
    "pop rock",
    "pop soul",
    "post-hardcore",
    "progressive folk rock",
    "progressive rock",
    "psychedelic",
    "psychedelic rock",
    "r&b",
    "rap",
    "rock",
    "rock n roll",
    "rockabilly",
    "screamo",
    "singer songwriter",
    "soft rock",
    "soul",
    "southern rock",
    "symphonic metal",
    "synthpop",
    "trap",
    "trip-hop"
}

GENRE_MAPPING = {
    # Hip hop
    "hip-hop": "hip hop",
    "hiphop": "hip hop",

    # R&B
    "rnb": "r&b",
    "r and b": "r&b",
    "rhythm and blues": "r&b",

    # Rock
    "rock and roll": "rock",
    "rock & roll": "rock",
    "rock n' roll": "rock n roll",

    # Electronic
    "edm": "electronic",
    "electronic dance music": "electronic",

    # Country
    "contry": "country",
    "country-pop": "country pop",
    "alt-country": "alt country",

    # Folk / rock variants
    "folkrock": "folk rock",
    "blues-rock": "blues rock",

    # Singer-songwriter
    "singer-songwriter": "singer songwriter",

    # Soul variants
    "neo soul": "neo-soul",

    # Trip hop
    "trip hop": "trip-hop"
}


def normalize_genre(genre):
    """
    Normaliza una etiqueta de género individual.
    Solo devuelve el género si pertenece a VALID_GENRES.
    Si no pertenece, devuelve None.
    """

    if pd.isna(genre):
        return None

    genre = str(genre)
    genre = html.unescape(genre)
    genre = genre.strip().lower()

    # Quitar comillas externas
    genre = genre.strip("'").strip('"')

    # Normalizar separadores simples
    genre = genre.replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip()

    # Quitar puntuación rara al principio/final, pero mantener símbolos internos como r&b o trip-hop
    genre = re.sub(r"^[^\w]+|[^\w]+$", "", genre)

    # Aplicar mapeo de variantes a una forma más estándar
    genre = GENRE_MAPPING.get(genre, genre)

    # Aceptar solo géneros válidos
    if genre not in VALID_GENRES:
        return None

    return genre


def parse_genre_cell(value):
    """
    Convierte una celda de género en una lista de géneros normalizados.
    Sirve para celdas simples, listas en string o valores separados por comas.
    Solo conserva géneros incluidos en VALID_GENRES.
    """

    if pd.isna(value):
        return []

    value = str(value).strip()

    parsed_values = []

    # Caso: "['pop', 'rock']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple, set)):
                parsed_values = list(parsed)
            else:
                parsed_values = [value]
        except Exception:
            parsed_values = [value]

    # Caso: "pop, rock, r&b"
    elif "," in value:
        parsed_values = value.split(",")

    # Caso simple: "pop"
    else:
        parsed_values = [value]

    normalized = []

    for genre in parsed_values:
        genre_norm = normalize_genre(genre)
        if genre_norm is not None:
            normalized.append(genre_norm)

    # Eliminar duplicados manteniendo el orden
    unique_normalized = []
    seen = set()

    for genre in normalized:
        if genre not in seen:
            unique_normalized.append(genre)
            seen.add(genre)

    return unique_normalized

In [170]:
# NORMALIZAR GENERO

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

display(df[["genres", "genre_1", "genre_2", "genre_3", 
            "genre_1_clean", "genre_2_clean", "genre_3_clean"]].head())

,genres,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean
0,"symphonic metal, female vocalists, female fron...",symphonic metal,female vocalists,female fronted metal,symphonic metal,NaN,NaN
1,"pop, love it, female vocalists",pop,love it,female vocalists,pop,NaN,NaN
2,"electropop, pop, synthpop",electropop,pop,synthpop,electropop,pop,synthpop
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**GENEROS POR CANCIÓN**

In [171]:
def build_genre_list(row):
    """
    Construye una lista de géneros únicos a partir de genre_1_clean,
    genre_2_clean y genre_3_clean.
    """
    
    genres = []
    
    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]
        if genre is not None and pd.notna(genre):
            genres.append(genre)
    
    # Eliminar duplicados manteniendo el orden
    unique_genres = []
    seen = set()
    
    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)
    
    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

display(df[["artist", "song", "genre_list", "n_genres"]].head())

,artist,song,genre_list,n_genres
0,Tarja Turunen,Until Silence,[symphonic metal],1
1,Susan Boyle,You Have To Be There,[pop],1
2,Taylor Swift,Cruel Summer,"[electropop, pop, synthpop]",3
3,Taylor Swift,London Boy,[],0
4,Taylor Swift,Sugar,[],0


In [172]:
MIN_WORDS = 30

initial_rows = len(df)

mask_valid_lyrics = df["lyrics_clean"].notna() & (df["word_count"] >= MIN_WORDS)
mask_valid_genres = df["n_genres"] > 0

df_clean = df[mask_valid_lyrics & mask_valid_genres].copy()

print("Filas iniciales:", initial_rows)
print("Filas tras limpieza:", len(df_clean))
print("Canciones eliminadas:", initial_rows - len(df_clean))

print("\nMotivos principales:")
print("Sin letra válida o letra demasiado corta:", (~mask_valid_lyrics).sum())
print("Sin género válido:", (~mask_valid_genres).sum())

Filas iniciales: 1337
Filas tras limpieza: 729
Canciones eliminadas: 608

Motivos principales:
Sin letra válida o letra demasiado corta: 481
Sin género válido: 234


In [173]:
def normalize_text_id(text):
    """
    Normaliza texto para detectar duplicados.
    No se usa para el modelo, solo para comparar artist/song.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    
    return text


df_clean["artist_norm"] = df_clean["artist"].apply(normalize_text_id)
df_clean["song_norm"] = df_clean["song"].apply(normalize_text_id)

duplicates = df_clean.duplicated(subset=["artist_norm", "song_norm"], keep=False)

print("Posibles duplicados:", duplicates.sum())

df_clean = (
    df_clean
    .sort_values("word_count", ascending=False)
    .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
    .sort_index()
    .copy()
)

print("Shape tras eliminar duplicados:", df_clean.shape)

Posibles duplicados: 0
Shape tras eliminar duplicados: (729, 16)


In [174]:
# Dataset expandido por genero

df_genres = df_clean.explode("genre_list").copy()

df_genres = df_genres.rename(columns={"genre_list": "genre"})

df_genres = df_genres[df_genres["genre"].notna()].copy()

# Evitar que una misma canción aparezca dos veces en el mismo género
df_genres = df_genres.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"],
    keep="first"
)

print("Shape df_clean, una fila por canción:", df_clean.shape)
print("Shape df_genres, una fila por canción-género:", df_genres.shape)

display(df_genres[["artist", "song", "genre", "lyrics_clean", "word_count"]].head())

Shape df_clean, una fila por canción: (729, 16)
Shape df_genres, una fila por canción-género: (1530, 16)


,artist,song,genre,lyrics_clean,word_count
1,Susan Boyle,You Have To Be There,pop,What is it Lord that you want That I am not se...,325
2,Taylor Swift,Cruel Summer,electropop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
2,Taylor Swift,Cruel Summer,pop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
2,Taylor Swift,Cruel Summer,synthpop,"(Yeah, yeah, yeah, yeah) Fever dream high in t...",447
5,Shania Twain,Forever And For Always,country,"Mm, mm Mm, in your arms Oh I can hear your hea...",389


In [175]:
genre_counts = df_genres["genre"].value_counts()


# 1. Le decimos a Pandas que no ponga límite máximo de filas
#pd.set_option('display.max_rows', None)

# 2. Mostramos tu tabla (sin el .head())
display(
    genre_counts
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

# 3. Restauramos el límite normal (por defecto suele ser 60)
# Esto es importante para que futuras tablas grandes no te bloqueen la pantalla
pd.reset_option('display.max_rows')

print("Número total de géneros únicos:", df_genres["genre"].nunique())
print("Número total de registros canción-género:", len(df_genres))


,n_songs,count
0,country,497
1,singer songwriter,130
2,classic country,110
3,folk,104
4,pop,88
...,...,...
85,pop soul,1
86,dream pop,1
87,neo-soul,1
88,symphonic metal,1


Número total de géneros únicos: 90
Número total de registros canción-género: 1530


In [176]:
MIN_SONGS_PER_GENRE = 1

valid_genres = genre_counts[genre_counts >= MIN_SONGS_PER_GENRE].index

df_genres_filtered = df_genres[df_genres["genre"].isin(valid_genres)].copy()

print("Géneros antes del filtro:", df_genres["genre"].nunique())
print("Géneros después del filtro:", df_genres_filtered["genre"].nunique())

print("Registros canción-género antes:", len(df_genres))
print("Registros canción-género después:", len(df_genres_filtered))

display(
    df_genres_filtered["genre"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

Géneros antes del filtro: 90
Géneros después del filtro: 90
Registros canción-género antes: 1530
Registros canción-género después: 1530


,n_songs,count
0,country,497
1,singer songwriter,130
2,classic country,110
3,folk,104
4,pop,88
...,...,...
85,pop soul,1
86,dream pop,1
87,neo-soul,1
88,symphonic metal,1


In [177]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_clean.to_csv(OUTPUT_DIR / "songs_clean_by_song.csv", index=False)
df_genres.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv", index=False)
df_genres_filtered.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded_filtered.csv", index=False)

print("Archivos guardados:")
print(OUTPUT_DIR / "songs_clean_by_song.csv")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded_filtered.csv")

Archivos guardados:
outputs/songs_clean_by_song.csv
outputs/songs_clean_by_genre_exploded.csv
outputs/songs_clean_by_genre_exploded_filtered.csv


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar las letras y los géneros musicales para el análisis emocional posterior. Se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido. La limpieza de las letras fue conservadora, evitando eliminar stopwords o modificar excesivamente el texto, ya que el modelo de emociones utilizado posteriormente necesita conservar el contexto lingüístico de las frases.

Además, se normalizaron las etiquetas de género presentes en las columnas genre_1, genre_2 y genre_3, convirtiéndolas a minúsculas, eliminando espacios innecesarios y unificando algunas variantes frecuentes. Dado que una canción puede pertenecer a varios géneros, se generó un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite analizar posteriormente la distribución emocional de las letras para cada género musical.

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `songs_clean_by_song.csv`                    | Una fila por canción        | Aplicar el modelo emocional     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |
| `songs_clean_by_genre_exploded_filtered.csv` | Canción-género, filtrado    | Gráficos y conclusiones fiables |


--- 

### 2. Usar genre_1 como análisis principal y, si hay tiempo, hacer una versión secundaria usando todos los géneros con formato “multi-label”.

---
### 3. Aplicar análisis de sentimiento a cada letra

Para cada canción se calcularía un sentimiento. El resultado ideal sería añadir nuevas columnas al dataset:

+ sentiment_score

+ sentiment_label

+ positive_score

+ neutral_score

+ negative_score

+ word_count


---
### 4.  Clasificación de canciones en sentimiento.

Buscar modelo potente:

+ Opción A: modelo de emociones con 7 clases --> Un modelo como j-hartmann/emotion-english-distilroberta-base clasifica texto en emociones básicas: anger, disgust, fear, joy, neutral, sadness y surprise.

+ Opción B: modelo GoEmotions con 28 emociones --> Un modelo como SamLowe/roberta-base-go_emotions está entrenado sobre el dataset GoEmotions y funciona como clasificación multi-label, es decir, una misma letra puede activar varias emociones a la vez.

---

### 5. Agregación por género.

Después de calcular el sentimiento por canción, se agrupa por género.

Métricas recomendadas por género:

+ n_canciones

+ sentimiento_medio

+ sentimiento_mediano

+ desviación_típica

+ % Sentimiento_1

+ % Sentimiento_2

+ % Sentimiento_X

---

### 6. Visualización resultados

+ Barras: sentimiento medio por género

+ Boxplot - Violinplot por género

+ Barras apiladas % sentimientos

+ Nº canciones por género